# SmartCampus AI — Phase A & B: Zero-Shot vs Few-Shot (+ Classic Dense RAG) Benchmark (<= 9B Open-Source)

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` (Account #2 for Qwen family, Account #3 for Gemma/Aya)

This notebook evaluates the 4 open-weight <= 9B finalists on the full 85-item stratified evaluation suite (`70` SA-32 cases in `evaluation_dataset.json` + `15` SA-34 dialect/code-switching stress cases in `dialect_stress_cases.json`) using **Stage 1 Classic Dense RAG (`BAAI/bge-m3`, `1024d`)** over the 42 real scraped official `https://iit.tn/` Markdown documents:
1. `Qwen/Qwen3-8B` (Apache 2.0, `enable_thinking=False` + `/no_think`)
2. `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0)
3. `google/gemma-2-9b-it` (Gemma Terms of Use)
4. `CohereForAI/aya-expanse-8b` (CC-BY-NC-4.0)


In [ ]:
# Cell 1 — Mount Google Drive (when on Colab) & install dependencies
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Running outside Google Colab; using local workspace paths.")

!pip install -q bitsandbytes accelerate transformers sentence-transformers pandas numpy


In [ ]:
# Cell 2 — Configuration & Path Resolution (change MODEL_ID per Colab Pro account run)
import json
import os
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"  # Swap to: Qwen/Qwen2.5-7B-Instruct | google/gemma-2-9b-it | CohereForAI/aya-expanse-8b
EMBEDDING_MODEL_ID = "BAAI/bge-m3"
ENABLE_THINKING = False
DRIVE_ROOT = Path("/content/drive/MyDrive/SmartCampus_AI")

def _discover_repo_root() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path("/content/Projet_IA_Generative"),
    ]
    for cand in candidates:
        if (cand / "backend" / "data" / "corpus" / "iit" / "manifest.json").exists():
            return cand
    return None

REPO_ROOT = _discover_repo_root()
OUTPUT_DIR = (DRIVE_ROOT / "benchmarks") if Path("/content/drive/MyDrive").exists() else Path("benchmarks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Target LLM:", MODEL_ID, "| enable_thinking:", ENABLE_THINKING)
print("Repo root:", REPO_ROOT or "Not cloned locally (will use Google Drive SmartCampus_AI/)")


In [ ]:
# Cell 3 — Load 42 Official Scraped IIT Corpus Docs + Markdown-Heading-Aware Chunking & BAAI/bge-m3 (1024d) Index
FRONTMATTER_RE = re.compile(r"^---\s*\n(.*?)\n---\s*\n", re.DOTALL)
HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$")

def strip_frontmatter_and_meta(raw_md: str) -> tuple[dict[str, str], str]:
    meta: dict[str, str] = {}
    match = FRONTMATTER_RE.match(raw_md)
    if not match:
        return meta, raw_md.strip()
    for line in match.group(1).splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            meta[k.strip()] = v.strip().strip('"').strip("'")
    return meta, raw_md[match.end():].strip()

def chunk_markdown_by_headings(
    md_body: str,
    *,
    default_section: str = "Overview",
    target_tokens: int = 512,
    overlap_tokens: int = 64,
) -> list[tuple[str, str]]:
    """Split Markdown by headings into ~512-token chunks with ~64-token overlap (~1.3 tokens/word)."""
    max_words = max(80, int(target_tokens / 1.3))
    overlap_words = max(16, int(overlap_tokens / 1.3))

    sections: list[tuple[str, list[str]]] = []
    current_heading = default_section
    current_lines: list[str] = []

    for line in md_body.splitlines():
        hm = HEADING_RE.match(line.strip())
        if hm:
            if current_lines:
                sections.append((current_heading, current_lines))
                current_lines = []
            current_heading = re.sub(r"[*_`]+", "", hm.group(2)).strip() or default_section
        else:
            current_lines.append(line)
    if current_lines:
        sections.append((current_heading, current_lines))

    chunks: list[tuple[str, str]] = []
    for heading, lines in sections:
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", "\n".join(lines)) if p.strip()]
        if not paragraphs:
            continue
        section_words = "\n\n".join(paragraphs).split()
        if len(section_words) <= max_words:
            chunks.append((heading, "\n\n".join(paragraphs)))
            continue
        step = max(1, max_words - overlap_words)
        for start in range(0, len(section_words), step):
            window = section_words[start : start + max_words]
            if not window:
                break
            chunks.append((heading, " ".join(window)))
            if start + max_words >= len(section_words):
                break
    return chunks

def load_official_iit_corpus() -> tuple[list[dict[str, str]], Path | None]:
    corpus_candidates = []
    if REPO_ROOT is not None:
        corpus_candidates.append(REPO_ROOT / "backend" / "data" / "corpus" / "iit")
    corpus_candidates.extend([
        Path("backend/data/corpus/iit"),
        Path("../../backend/data/corpus/iit"),
        DRIVE_ROOT / "corpus" / "iit",
    ])

    corpus_dir = next((p for p in corpus_candidates if p.exists() and any(p.glob("*.md"))), None)
    loaded_chunks: list[dict[str, str]] = []

    if corpus_dir is not None:
        manifest_path = corpus_dir / "manifest.json"
        if manifest_path.exists():
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
            doc_entries = manifest.get("documents", [])
        else:
            doc_entries = [{"file_name": p.name} for p in sorted(corpus_dir.glob("*.md"))]

        for entry in doc_entries:
            md_path = corpus_dir / entry["file_name"]
            if not md_path.exists():
                continue
            fm, body = strip_frontmatter_and_meta(md_path.read_text(encoding="utf-8"))
            title = entry.get("title") or fm.get("title") or md_path.stem
            source_url = entry.get("source_url") or fm.get("source_url") or md_path.name
            school = entry.get("school") or fm.get("school") or "iit"
            doc_type = entry.get("doc_type") or fm.get("doc_type") or "general_brochure"
            code = entry.get("code") or fm.get("code") or md_path.stem
            for idx, (sec, txt) in enumerate(chunk_markdown_by_headings(body, default_section=title)):
                loaded_chunks.append({
                    "chunk_id": f"{code}#c{idx}",
                    "code": code,
                    "title": title,
                    "source": source_url,
                    "section": sec,
                    "school": school,
                    "doc_type": doc_type,
                    "text": f"{title} — {sec}\n{txt}",
                })
        print(f"Loaded {len(loaded_chunks)} chunks from {len(doc_entries)} official docs in: {corpus_dir}")
        return loaded_chunks, corpus_dir

    # Minimal verified fallback ONLY when neither the local repo nor Google Drive SmartCampus_AI/corpus/iit/ exists
    print(
        "WARNING: Official corpus directory not found locally or in Google Drive "
        f"({DRIVE_ROOT / 'corpus' / 'iit'}). Upload backend/data/corpus/iit/ to Drive for full 42-doc RAG."
    )
    fallback_chunks = [
        {
            "chunk_id": "IIT-WEB-FILIERES-INGENIEUR-GENIE-INFORMATIQUE-2#fallback",
            "code": "IIT-WEB-FILIERES-INGENIEUR-GENIE-INFORMATIQUE-2",
            "title": "Cycle Ingénieur Génie Informatique à Sfax | IIT",
            "source": "https://iit.tn/filieres-ingenieur/genie-informatique-2/",
            "section": "Organisation du diplôme",
            "school": "iit",
            "doc_type": "programme",
            "text": (
                "Le Groupe Universitaire NAU (IIT Sfax) propose un diplôme d'ingénieur en Génie Informatique "
                "habilité par le Ministère de l'Enseignement Supérieur et accrédité ASIIN / Euro-Inf (niveau Master), "
                "avec trois parcours : GLID (Génie Logiciel et Informatique Décisionnelle), ARSI (Administration Réseau "
                "et Sécurité Informatique) et SDIA (Science des Données et Intelligence Artificielle). "
                "Pré-requis : Prépa, Licence ou Mastère."
            ),
        },
        {
            "chunk_id": "IIT-WEB-PROCEDURE-ET-FRAIS-DINSCRIPTION#fallback",
            "code": "IIT-WEB-PROCEDURE-ET-FRAIS-DINSCRIPTION",
            "title": "Frais d'Inscription IIT Sfax | Procédure d'Admission",
            "source": "https://iit.tn/test/procedure-et-frais-dinscription/",
            "section": "Procédure d'admission",
            "school": "iit",
            "doc_type": "fees_schedule",
            "text": (
                "La candidature aux filières d'ingénierie de l'IIT s'effectue par préinscription en ligne, "
                "concours sur dossier, entretien de validation et test d'admission devant une commission "
                "d'enseignants permanents. À l'issue de l'entretien, seule la commission / NAU Responsable "
                "décide de l'acceptation, du refus ou de la mise en liste d'attente. Les tarifs officiels "
                "à jour sont confirmés par le secrétariat administratif IIT / NAU."
            ),
        },
    ]
    return fallback_chunks, None

OFFICIAL_CHUNKS, ACTIVE_CORPUS_DIR = load_official_iit_corpus()
embedder = SentenceTransformer(EMBEDDING_MODEL_ID)
chunk_vecs = embedder.encode(
    [c["text"] for c in OFFICIAL_CHUNKS],
    normalize_embeddings=True,
    show_progress_bar=False,
)
assert chunk_vecs.shape[1] == 1024, f"Expected BAAI/bge-m3 1024d vectors, got {chunk_vecs.shape[1]}d"
print(f"Indexed {len(OFFICIAL_CHUNKS)} chunks | Embedding dimension: {chunk_vecs.shape[1]}d")

def retrieve_classic_rag(query: str, top_k: int = 4):
    q_vec = embedder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    scores = chunk_vecs @ q_vec
    idxs = np.argsort(scores)[::-1][:top_k]
    return [(OFFICIAL_CHUNKS[int(i)], float(scores[int(i)])) for i in idxs]


In [ ]:
# Cell 4 — Load 4-bit LLM cleanly on L4 / T4 & Align VISION_SYSTEM_PROMPT with backend/app/services/llm_service.py
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
)

# Verbatim alignment with backend/app/services/llm_service.py::VISION_SYSTEM_PROMPT
VISION_SYSTEM_PROMPT = (
    "You are VISION, the conversational academic orientation and pre-registration supervisor "
    "for SmartCampus AI covering the IIT and ISB schools (plus the announced ISA extension) "
    "within the NAU group in Sfax, Tunisia. You welcome any Visitor (unauthenticated visitor: "
    "high-school student, university student, or parent/relative inquiring on behalf of a "
    "prospective candidate) as well as any authenticated Candidate.\n\n"
    "STRICT AND NON-NEGOTIABLE RULES (SC V3):\n"
    "1. Multilingualism: Always reply in the exact language or register used by the Visitor "
    "or Candidate (French, Modern Standard Arabic / Fusha, Tunisian Derja in Arabic script, or "
    "natural Tunisian Derja / Arabizi for Arabizi and FR-TN code-switching queries). "
    "NEVER reply in Classical/Modern Standard Arabic (Fusha) to a Derja or Arabizi message.\n"
    "2. Zero hallucination of academic facts: NEVER invent tuition fees, admission thresholds, "
    "deadlines, guaranteed scholarships, or non-existent programs (e.g., medicine, pharmacy, "
    "law degree, astrophysics, criminology, agricultural/veterinary engineering). If an "
    "official fact is not provided in the context, state honestly that it must be verified "
    "in the official NAU catalog managed by the school Responsable.\n"
    "3. Human-in-the-Loop (HITL) Governance: You may NEVER admit, reject, waitlist, promise "
    "admission, or issue an official certificate. Only the human NAU Responsable makes and "
    "signs the final admission decision.\n"
    "4. 'Give or Keep' Rule (Visitor -> Candidate): Orientation and Q&A are 100% open to "
    "any Visitor without an account (7-day anonymous temporary session). Authentication "
    "(OTP or Google OAuth, which automatically merges the session and promotes the Visitor "
    "to Candidate) is required only when uploading an official grade transcript, saving a "
    "shortlist, booking an interview slot, or submitting a pre-registration.\n"
    "5. Dialect / Arabizi ambiguity fallback: If a message is too ambiguous, abbreviated, or "
    "noisy to identify the Visitor's or Candidate's intent reliably, never guess; politely "
    "ask them to rephrase their question in simple French.\n"
    "6. Style: Be concise, structured, and helpful, and systematically cite the official "
    "source documents provided in the context using [Source: ...].\n"
    "7. Administrative rigor and declarative vs. official separation: NEVER present "
    "unverified, speculative, or self-declared administrative information as official facts. "
    "Any information about tuition fees, scholarships, admission thresholds, or equivalences "
    "must come exclusively from the provided official RAG catalog excerpts or structured DB "
    "records maintained by the Responsable. When no formal official source is in the context, "
    "explicitly remind the Visitor or Candidate that the information must be confirmed by "
    "the NAU Responsable or administrative secretariat."
)

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "3aslema, nheb na3ref chnowa les spécialités fel cycle ingénieur génie informatique fi l'IIT?",
    },
    {
        "role": "assistant",
        "content": (
            "3aslema! Hasb el catalogue el rasmi [Source: https://iit.tn/filieres-ingenieur/genie-informatique-2/, §Organisation], "
            "el Cycle Ingénieur Génie Informatique fi l'IIT Sfax (accrédité ASIIN / Euro-Inf) fih 3 parcours de spécialisation : "
            "GLID (Génie Logiciel et Informatique Décisionnelle), ARSI (Administration Réseau et Sécurité Informatique) "
            "w SDIA (Science des Données et Intelligence Artificielle). Tnajem t9olli chnowa diplômek tawa (Prépa, Licence walla Mastère) bech nwajhek akther?"
        ),
    },
    {
        "role": "user",
        "content": "Est-ce que je suis automatiquement admis si j'ai 15 de moyenne, et faites-vous Médecine ?",
    },
    {
        "role": "assistant",
        "content": (
            "Non : d'une part l'IIT, l'ISB et l'ISA ne proposent pas de filière Médecine dans le catalogue officiel NAU, "
            "et d'autre part [Source: https://iit.tn/test/procedure-et-frais-dinscription/, §Procédure d’admission] "
            "aucune admission n'est automatique — l'admission se fait sur dossier, entretien de validation et test, "
            "et seul le NAU Responsable / la commission humaine prend et signe la décision finale."
        ),
    },
]

THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)

def _needs_no_think(model_id: str) -> bool:
    lower = model_id.lower()
    return (not ENABLE_THINKING) and any(tag in lower for tag in ("qwen3", "smartcampus-vision"))

def _apply_chat_template_safe(messages: list[dict[str, str]]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING
        )
    except TypeError:
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            # Models such as Gemma-2 without a dedicated system role: fold system prompt into first user turn
            sys_txt = "\n\n".join(m["content"] for m in messages if m["role"] == "system")
            merged = [dict(m) for m in messages if m["role"] != "system"]
            if merged and merged[0]["role"] == "user":
                merged[0]["content"] = f"{sys_txt}\n\n{merged[0]['content']}"
            return tokenizer.apply_chat_template(merged, tokenize=False, add_generation_prompt=True)

def generate_answer(query: str, *, mode: str = "zero_shot", top_k: int = 4) -> tuple[str, float, list[tuple[dict, float]]]:
    hits = retrieve_classic_rag(query, top_k=top_k)
    ctx = "\n".join(
        f"[Source: {c['source']}, §{c['section']}] (score={s:.2f}): {c['text']}"
        for c, s in hits
    )
    system_content = f"{VISION_SYSTEM_PROMPT}\n\n[OFFICIAL RAG EXCERPTS]\n{ctx}"
    if _needs_no_think(MODEL_ID) and "/no_think" not in system_content:
        system_content = f"{system_content}\n/no_think"

    messages = [{"role": "system", "content": system_content}]
    if mode == "few_shot":
        messages.extend(FEW_SHOT_EXAMPLES)
    messages.append({"role": "user", "content": query})

    prompt = _apply_chat_template_safe(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=320,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_s = round(time.perf_counter() - t0, 3)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    cleaned = THINK_RE.sub("", raw).strip()
    return cleaned, latency_s, hits


In [ ]:
# Cell 5 — Load Real 85-Item Evaluation Suite (70 SA-32 + 15 SA-34) & Export CSV to Google Drive
MAX_EVAL_SAMPLES: int | None = None  # Set e.g. 10 for a quick smoke slice, or None for all 85 cases

def load_evaluation_suite(max_samples: int | None = None) -> list[dict]:
    data_candidates = []
    if REPO_ROOT is not None:
        data_candidates.append(REPO_ROOT / "backend" / "tests" / "data")
    data_candidates.extend([
        Path("backend/tests/data"),
        Path("../../backend/tests/data"),
        DRIVE_ROOT / "data",
    ])

    items: list[dict] = []
    eval_file = next((d / "evaluation_dataset.json" for d in data_candidates if (d / "evaluation_dataset.json").exists()), None)
    stress_file = next((d / "dialect_stress_cases.json" for d in data_candidates if (d / "dialect_stress_cases.json").exists()), None)

    if eval_file is not None:
        sa32 = json.loads(eval_file.read_text(encoding="utf-8"))
        for row in sa32:
            items.append({
                "suite": "SA-32",
                "id": row["id"],
                "language_group": row.get("language_group", "unknown"),
                "intent": row.get("intent", ""),
                "target_school": row.get("target_school", ""),
                "expected_language": row.get("expected_language", ""),
                "query": row["query"],
                "ground_truth": row.get("ground_truth", ""),
                "expected_fallback": False,
            })
        print(f"Loaded {len(sa32)} SA-32 cases from: {eval_file}")

    if stress_file is not None:
        sa34 = json.loads(stress_file.read_text(encoding="utf-8"))
        for row in sa34:
            items.append({
                "suite": "SA-34",
                "id": row["id"],
                "language_group": row.get("language_group", "unknown"),
                "intent": row.get("intent", ""),
                "target_school": row.get("target_school", ""),
                "expected_language": row.get("expected_language", ""),
                "query": row["query"],
                "ground_truth": row.get("notes", ""),
                "expected_fallback": bool(row.get("expected_fallback", False)),
            })
        print(f"Loaded {len(sa34)} SA-34 dialect stress cases from: {stress_file}")

    if not items:
        raise FileNotFoundError(
            "Could not locate evaluation_dataset.json or dialect_stress_cases.json. "
            f"Clone the repository or copy both files from backend/tests/data/ into {DRIVE_ROOT / 'data'}."
        )

    if max_samples is not None:
        items = items[:max_samples]
    print(f"Running benchmark on {len(items)} evaluation cases (MAX_EVAL_SAMPLES={max_samples})")
    return items

EVAL_ITEMS = load_evaluation_suite(MAX_EVAL_SAMPLES)

rows = []
for mode in ("zero_shot", "few_shot"):
    for item in EVAL_ITEMS:
        ans, lat, hits = generate_answer(item["query"], mode=mode)
        top_chunk, top_score = hits[0]
        rows.append({
            "model": MODEL_ID,
            "mode": mode,
            "suite": item["suite"],
            "id": item["id"],
            "language_group": item["language_group"],
            "intent": item["intent"],
            "target_school": item["target_school"],
            "expected_language": item["expected_language"],
            "expected_fallback": item["expected_fallback"],
            "latency_s": lat,
            "has_citation": "[Source:" in ans,
            "top_rag_source": top_chunk["source"],
            "top_rag_score": round(top_score, 4),
            "query": item["query"],
            "answer": ans,
            "ground_truth": item["ground_truth"],
        })

df = pd.DataFrame(rows)
slug = MODEL_ID.replace("/", "__")
csv_path = OUTPUT_DIR / f"benchmark_{slug}.csv"
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)
summary = (
    df.groupby(["mode", "suite", "language_group"], as_index=False)
    .agg(
        cases=("id", "count"),
        mean_latency_s=("latency_s", "mean"),
        citation_rate=("has_citation", "mean"),
        mean_top_rag_score=("top_rag_score", "mean"),
    )
)
summary
